# 04 — GitHub Community Usage Evaluation

The core methodological contribution: evaluating translation quality not by
back-translation against a reference (which assumes a ground truth exists)
but by comparing pipeline outputs to how scholarly communities actually name
these concepts in their practice.

This notebook analyses the GitHub alignment results and builds the evidence
for the paper's argument that:
  1. Pipeline outputs do not reliably reflect community usage for lower-resource
     language communities
  2. Some communities have adopted the English term as a loan word, which is a
     meaningful community choice that the pipeline cannot detect
  3. The distribution of GitHub signal across languages is itself evidence about
     which communities are represented in global DH discourse

**Run order:** After `evaluate_github_alignment.py`.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import altair as alt
from pathlib import Path

alt.data_transformers.enable('vegafusion')

sys.path.insert(0, str(Path('..').resolve()))
from scripts.utils import get_data_directory_path, read_csv_file

DATA_DIR = get_data_directory_path()
EVAL_DIR = os.path.join(DATA_DIR, 'translated_terms', 'digital_humanities', 'evaluation')
TARGET_TERMS = ['Digital Humanities']
VARIANTS = ['minimal', 'expert_persona', 'native_rationale', 'judge']

VARIANT_LABELS = {
    'minimal':          'Minimal',
    'expert_persona':   'Expert Persona',
    'native_rationale': 'Native Rationale',
    'judge':            'Judge',
}
VARIANT_COLOURS = {
    'minimal':          '#DD8452',
    'expert_persona':   '#55A868',
    'native_rationale': '#8172B2',
    'judge':            '#4C72B0',
}
SIGNAL_COLOURS = {
    'strong': '#2ca02c',
    'weak':   '#98df8a',
    'loan':   '#ff7f0e',
    'absent': '#d62728',
    'no_translation': '#aec7e8',
}

# Load evaluation outputs
align_path   = os.path.join(EVAL_DIR, 'github_alignment.csv')
summary_path = os.path.join(EVAL_DIR, 'github_alignment_summary.csv')
absent_path  = os.path.join(EVAL_DIR, 'absent_concept_candidates.csv')
loan_path    = os.path.join(EVAL_DIR, 'loan_word_cases.csv')

align_df   = read_csv_file(align_path)   if os.path.exists(align_path)   else None
summary_df = read_csv_file(summary_path) if os.path.exists(summary_path) else None
absent_df  = read_csv_file(absent_path)  if os.path.exists(absent_path)  else None
loan_df    = read_csv_file(loan_path)    if os.path.exists(loan_path)    else None

for name, df in [('Alignment', align_df), ('Summary', summary_df),
                  ('Absent', absent_df), ('Loan words', loan_df)]:
    print(f'{name}: {len(df) if df is not None else "NOT FOUND"} rows')

## 4.1 Overall Signal Distribution

How many languages fall into each alignment category?

In [ ]:
if align_df is not None:
    signal_col = 'minimal_signal'
    if signal_col in align_df.columns:
        counts = align_df[signal_col].value_counts().reset_index()
        counts.columns = ['signal', 'count']
        total = counts['count'].sum()
        counts['pct'] = counts['count'] / total * 100
        counts['color'] = counts['signal'].map(SIGNAL_COLOURS).fillna('grey')

        print('Signal distribution (minimal variant):')
        for _, r in counts.iterrows():
            bar = '█' * int(r['count'] / total * 40)
            print(f'  {r["signal"]:15s} {bar} {int(r["count"]):3d} ({r["pct"]:.1f}%)')

        arc = alt.Chart(counts).mark_arc(outerRadius=130).encode(
            theta=alt.Theta('count:Q'),
            color=alt.Color('signal:N', scale=alt.Scale(
                domain=list(SIGNAL_COLOURS.keys()),
                range=list(SIGNAL_COLOURS.values()),
            )),
            tooltip=['signal', 'count', alt.Tooltip('pct:Q', format='.1f')],
        )
        text = arc.mark_text(radius=155, size=12).encode(
            text=alt.Text('signal:N'),
        )
        (arc + text).properties(
            title='GitHub Alignment Signal Distribution (Minimal Variant)',
            width=350, height=350,
        ).display()
    else:
        print(f'Column {signal_col} not found. Available:', list(align_df.columns[:10]))

## 4.2 Signal Distribution by Language Family

Does alignment with community usage vary systematically by language family?
Expected finding: higher alignment for Romance/Germanic (over-represented in
DH discourse) and lower for African, South/Southeast Asian language families.

In [ ]:
if summary_df is not None:
    minimal_summary = summary_df[summary_df['variant'] == 'minimal'] \
                      if 'variant' in summary_df.columns else summary_df

    if len(minimal_summary) > 0:
        signal_cols = ['strong_match', 'weak_match', 'loan_word', 'absent']
        available = [c for c in signal_cols if c in minimal_summary.columns]

        if available and 'language_family' in minimal_summary.columns:
            long_df = minimal_summary.melt(
                id_vars='language_family', value_vars=available,
                var_name='signal', value_name='count',
            )
            long_df['signal_label'] = long_df['signal'].str.replace('_', ' ').str.title()

            alt.Chart(long_df).mark_bar().encode(
                x=alt.X('language_family:N', title='Language Family'),
                y=alt.Y('count:Q', title='Number of Languages'),
                color=alt.Color('signal:N', scale=alt.Scale(
                    domain=['strong_match', 'weak_match', 'loan_word', 'absent'],
                    range=[SIGNAL_COLOURS['strong'], SIGNAL_COLOURS['weak'],
                           SIGNAL_COLOURS['loan'], SIGNAL_COLOURS['absent']],
                )),
                order=alt.Order('signal:N'),
                tooltip=['language_family', 'signal_label', 'count'],
            ).properties(
                title='GitHub Alignment by Language Family (Minimal Variant)',
                width=500, height=300,
            ).display()
        else:
            print('Summary columns:', list(minimal_summary.columns))

## 4.3 Does Variant Choice Affect GitHub Alignment?

The key question: does the epistemic stance in the prompt produce translations
that are more or less aligned with community practice?

In [ ]:
if align_df is not None:
    rows = []
    for v in VARIANTS:
        col = f'{v}_signal'
        if col not in align_df.columns:
            continue
        counts = align_df[col].value_counts(normalize=True).mul(100)
        for signal, pct in counts.items():
            rows.append({'variant': v, 'label': VARIANT_LABELS.get(v, v), 'signal': signal, 'pct': pct})

    if rows:
        comparison_df = pd.DataFrame(rows)
        signal_order = ['strong', 'weak', 'loan', 'absent', 'no_translation']

        alt.Chart(comparison_df).mark_bar().encode(
            x=alt.X('label:N', sort=list(VARIANT_LABELS.values()), title='Prompt Variant'),
            y=alt.Y('pct:Q', title='% of languages'),
            color=alt.Color('signal:N', sort=signal_order, scale=alt.Scale(
                domain=signal_order,
                range=[SIGNAL_COLOURS[s] for s in signal_order],
            )),
            order=alt.Order('signal:N'),
            tooltip=['label', 'signal', alt.Tooltip('pct:Q', format='.1f')],
        ).properties(
            title='GitHub Alignment Rate by Prompt Variant',
            width=450, height=300,
        ).display()

        print('\nSignal distribution by variant (%):')
        pivot = comparison_df.pivot_table(index='label', columns='signal', values='pct', fill_value=0)
        print(pivot.round(1).to_string())

## 4.4 Deep Dive: Languages with GitHub Data

For languages where we have substantial GitHub data, examine specific cases
where pipeline translations match or diverge from community usage.

In [ ]:
if align_df is not None and 'github_text_count' in align_df.columns:
    # Languages with most GitHub data
    data_rich = align_df.sort_values('github_text_count', ascending=False).head(20)
    print('Languages with most GitHub data:')
    display_cols = ['language_code', 'language_name', 'github_text_count',
                    'minimal_signal', 'minimal_translation']
    available = [c for c in display_cols if c in data_rich.columns]
    print(data_rich[available].to_string(index=False))

    # Strong match cases: what did the pipeline correctly predict?
    strong = align_df[align_df.get('minimal_signal', pd.Series()) == 'strong']
    if len(strong) > 0:
        print(f'\nStrong match cases ({len(strong)}):')
        ev_col = 'minimal_evidence'
        for _, row in strong.head(10).iterrows():
            lang = row.get('language_name', row.get('language_code', '?'))
            trans = row.get('minimal_translation', '?')
            evid = row.get(ev_col, '') if ev_col in row else ''
            print(f'  {lang}: "{trans}" → {evid[:80] if evid else "(no evidence snippet)"}')

## 4.5 The GitHub Signal as a DH Presence Map

The distribution of GitHub data across languages is itself evidence about
which communities are present in global DH discourse — separate from whether
their translations are correct. Visualise as a proportional map.

In [ ]:
if align_df is not None and 'github_text_count' in align_df.columns:
    LANGUAGE_FAMILIES = {
        'Romance':['es','fr','it','pt','ro','ca'], 'Germanic':['de','nl','sv','da','no','af'],
        'Slavic':['ru','pl','cs','sk','bg','hr','sr','uk'], 'East Asian':['zh','ja','ko'],
        'Semitic':['ar','he'], 'South Asian':['hi','bn','ur','ta','te'],
    }
    def get_family(code):
        for f, codes in LANGUAGE_FAMILIES.items():
            if code in codes: return f
        return 'Other'

    align_df['language_family'] = align_df['language_code'].apply(get_family)
    github_by_family = align_df.groupby('language_family')['github_text_count'].agg(
        total='sum', mean='mean', n='count'
    ).reset_index()

    print('GitHub data presence by language family:')
    print(github_by_family.round(1).to_string(index=False))

    total_chart = alt.Chart(github_by_family).mark_bar(color='steelblue').encode(
        x=alt.X('language_family:N', sort='-y', title='Language Family'),
        y=alt.Y('total:Q', title='Total text count'),
        tooltip=['language_family', 'total', 'n'],
    ).properties(title='Total GitHub Repo Texts by Language Family', width=300, height=250)

    mean_chart = alt.Chart(github_by_family).mark_bar(color='darkorange').encode(
        x=alt.X('language_family:N', sort='-y', title='Language Family'),
        y=alt.Y('mean:Q', title='Mean texts per language'),
        tooltip=['language_family', alt.Tooltip('mean:Q', format='.1f'), 'n'],
    ).properties(title='Mean GitHub Texts per Language', width=300, height=250)

    (total_chart | mean_chart).properties(
        title='DH Presence on GitHub by Language Family'
    ).display()

## 4.6 Summary Table for Paper

A clean summary table suitable for inclusion in the DHQ article.

In [ ]:
if align_df is not None:
    paper_rows = []
    for v in VARIANTS:
        col = f'{v}_signal'
        if col not in align_df.columns: continue
        total = len(align_df)
        counts = align_df[col].value_counts()
        paper_rows.append({
            'Variant': v.replace('_', ' ').title(),
            'N Languages': total,
            'Strong Match (%)': round(counts.get('strong', 0) / total * 100, 1),
            'Weak Match (%)':   round(counts.get('weak', 0)   / total * 100, 1),
            'Loan Word (%)':    round(counts.get('loan', 0)   / total * 100, 1),
            'Absent (%)':       round(counts.get('absent', 0) / total * 100, 1),
            'No Translation (%)': round(counts.get('no_translation', 0) / total * 100, 1),
        })
    
    paper_table = pd.DataFrame(paper_rows)
    print('Table for paper:')
    print(paper_table.to_string(index=False))
    
    paper_table.to_csv(os.path.join(EVAL_DIR, '04_paper_summary_table.csv'), index=False)
    print(f'\nSaved to {EVAL_DIR}/04_paper_summary_table.csv')